# 01 — Exploratory Data Analysis (EDA)

Цель: загрузка, очистка, feature engineering и визуализация датасета **Student Depression Dataset**.

Весь код предобработки вынесен в `src/preprocessing.py`; в этом ноутбуке мы вызываем его функции и анализируем результаты.

## 0. Настройка окружения

In [ ]:
import sys
from pathlib import Path

# Добавляем корень проекта в sys.path, чтобы импортировать src/
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f'Корень проекта: {project_root}')

In [ ]:
from src.preprocessing import (
    load_raw_data,
    clean_data,
    engineer_features,
    run_all_plots,
    split_and_save,
    get_paths,
)

paths = get_paths(project_root)
images_dir = paths['images_dir']
images_dir.mkdir(parents=True, exist_ok=True)

## 1. Загрузка данных

Исходный файл должен лежать по адресу `data/raw/Student Depression Dataset.csv`.

In [ ]:
df = load_raw_data(project_root)
df.head()

## 2. Описание датасета

In [ ]:
print(f'Размер датасета: {df.shape[0]} строк, {df.shape[1]} столбцов')
print('\nОбщая информация:')
df.info()
print('\nСтатистика:')
display(df.describe())
print('\nРаспределение целевой переменной:')
display(df['Depression'].value_counts(normalize=True).rename('доля'))

## 3. Очистка данных

Шаги: удаление пропусков → удаление дубликатов → приведение типов.

In [ ]:
print('Пропуски до очистки:')
display(df.isnull().sum()[df.isnull().sum() > 0])

df = clean_data(df)

print('\nПропуски после очистки:', df.isnull().sum().sum())

## 4. Feature Engineering

Создаём два новых признака:
- **Overall_Pressure** = Academic Pressure + Work Pressure — суммарная нагрузка;
- **Satisfaction_Ratio** = Study Satisfaction / (Job Satisfaction + ε) — соотношение удовлетворённости.

In [ ]:
df = engineer_features(df)
df[['Overall_Pressure', 'Satisfaction_Ratio']].describe()

## 5. Визуализации

In [ ]:
# Все графики сохраняются в report/images/
run_all_plots(df, save_dir=images_dir)

## 6. Разделение данных и сохранение

Стратегия сплита **Train 70% / Val 15% / Test 15%** с стратификацией по целевой переменной.

**Предотвращение data leakage:** сплит выполняется до любого кодирования признаков; `ColumnTransformer` в `modeling.py` обучается только на Train.

In [ ]:
train_df, val_df, test_df = split_and_save(df, project_root=project_root)

print('\nРаспределение целевой переменной в сплитах:')
for name, split in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    ratio = split['Depression'].mean()
    print(f'  {name}: {len(split)} строк | депрессия = {ratio:.2%}')